In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import os
import sys
sys.path.append("/home/tmp/packages")
from torch.nn import LayerNorm, Linear, Dropout, Softmax
from einops import rearrange, repeat
import copy
from timm.layers import DropPath, trunc_normal_
from pathlib import Path
import re
import torch.backends.cudnn as cudnn
import record
import matplotlib.pyplot as plt
from torchinfo import summary
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report, cohen_kappa_score
from operator import truediv
import math
from PIL import Image
import time
import torchvision.transforms.functional as TF
from torch.nn.parameter import Parameter
from sklearn.decomposition import PCA
from scipy.io import loadmat as loadmat
from scipy import io
import torch.utils.data as dataf
import torch.nn as nn
import torch
import torch.nn.functional as F
from torch import einsum
import random
import numpy as np
import os
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
from torch.optim.lr_scheduler import CosineAnnealingLR
from thop import profile
import h5py 
cudnn.deterministic = True
cudnn.benchmark = False

# create model

In [2]:
from torch.nn import LayerNorm,Linear,Dropout,Softmax
import copy
import torch.fft

def INF(B,H,W):
     return -torch.diag(torch.tensor(float("inf")).cuda().repeat(H),0).unsqueeze(0).repeat(B*W,1,1)

     
class HetConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1,padding = None, bias = None,p = 64, g = 64):
        super(HetConv, self).__init__() 
        # Groupwise Convolution
        self.gwc = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size,groups=g,padding = kernel_size//3, stride = stride)
        # Pointwise Convolution
        self.pwc = nn.Conv2d(in_channels, out_channels, kernel_size=1,groups=p, stride = stride)
    def forward(self, x):
        return self.gwc(x) + self.pwc(x)   
    
class FrequencyEnhancer(nn.Module):
    def __init__(self, dim, h=11, w=11):
        super().__init__()
        self.dim = dim
        self.h = h
        self.w = w
        
        self.freq_h = h
        self.freq_w = w // 2 + 1
        
        self.complex_weight = nn.Parameter(
            torch.randn(dim, self.freq_h, self.freq_w, 2, dtype=torch.float32) * 0.02
        )

        self.lidar_modulator = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(dim, dim),
            nn.Sigmoid() 
        )

    def forward(self, x, x_lidar):
        x_freq = torch.fft.rfft2(x, norm='ortho')
        weight = torch.view_as_complex(self.complex_weight)
        
        scale = self.lidar_modulator(x_lidar) # [B, dim]
        scale = scale.view(-1, self.dim, 1, 1) 
        
        dynamic_weight = weight * scale
        
        x_freq_enhanced = x_freq * dynamic_weight
        
        x_out = torch.fft.irfft2(x_freq_enhanced, s=(self.h, self.w), norm='ortho')
        
        return x + x_out    
    
class DualModalBoundaryLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=1.0, classes=15, weight=None): 
        super(DualModalBoundaryLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.ce_loss = nn.CrossEntropyLoss(weight=weight,label_smoothing=0.2, reduction='none') 
        #define sobel operator
        kernel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3) 
        kernel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer('kernel_x', kernel_x)
        self.register_buffer('kernel_y', kernel_y)

    def get_spatial_gradient(self, img):  # img: [B, C, H, W],gradient: [B, 1, H, W]
        if img.shape[1] > 1:
            img = torch.mean(img, dim=1, keepdim=True)

        gx = F.conv2d(img, self.kernel_x, padding=1)
        gy = F.conv2d(img, self.kernel_y, padding=1)
        gradient = torch.sqrt(gx**2 + gy**2)
        return gradient

    def forward(self, logits, target, hsi_img, lidar_img):  #logits:[B, Classes],target:[B],hsi_img:[B, C_hsi, H, W],lidar_img: [B, C_lidar, H, W]
        base_loss = self.ce_loss(logits, target)

        with torch.no_grad():
            g_hsi = self.get_spatial_gradient(hsi_img).detach()
            g_lidar = self.get_spatial_gradient(lidar_img).detach()

            h_center = hsi_img.shape[2] // 2
            w_center = hsi_img.shape[3] // 2
            
            w_hsi = g_hsi[:, 0, h_center, w_center]     # [B]
            w_lidar = g_lidar[:, 0, h_center, w_center] # [B]
            
            w_hsi = w_hsi / (w_hsi.max() + 1e-6)
            w_lidar = w_lidar / (w_lidar.max() + 1e-6)
            final_weight = 1.0 + self.beta * w_hsi + self.alpha * w_lidar
    
        weighted_loss = (base_loss * final_weight).mean()    
        return weighted_loss    

class DynamicMlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)

        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        if self.fc1.bias is not None:
            nn.init.normal_(self.fc1.bias, std=1e-6)
        if self.fc2.bias is not None:
            nn.init.normal_(self.fc2.bias, std=1e-6)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class CrossMambaMixer(nn.Module):
    def __init__(
        self,
        d_model,            
        d_state=16,         
        d_conv=4,           
        expand=2,           
        dt_rank="auto",     
        dt_min=0.001,
        dt_max=0.1,
        dt_init="random",
        dt_scale=1.0,
        dt_init_floor=1e-4,
        conv_bias=True,
        bias=False,
        use_fast_path=True, 
        layer_idx=None,
        device=None,
        dtype=None,
    ):
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model) #64 -> 128
        self.dt_rank = math.ceil(self.d_model / 16) if dt_rank == "auto" else dt_rank
        self.use_fast_path = use_fast_path
        self.layer_idx = layer_idx

        #HSI
        self.in_proj = nn.Linear(self.d_model, self.d_inner * 2, bias=bias, **factory_kwargs)

        #x branch
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            bias=conv_bias,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,  
            **factory_kwargs,
        )

        #compute SSM parameters
        self.x_proj = nn.Linear(
            self.d_inner, self.dt_rank + self.d_state * 2, bias=False, **factory_kwargs
        )
        
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True, **factory_kwargs)

        #Lidar
        self.lidar_modulator = nn.Linear(self.d_model, self.dt_rank, bias=False, **factory_kwargs)

        dt_init_std = self.dt_rank**-0.5 * dt_scale
        if dt_init == "constant":
            nn.init.constant_(self.dt_proj.weight, dt_init_std)
        elif dt_init == "random":
            nn.init.uniform_(self.dt_proj.weight, -dt_init_std, dt_init_std)
        
        dt = torch.exp(
            torch.rand(self.d_inner, **factory_kwargs) * (math.log(dt_max) - math.log(dt_min))
            + math.log(dt_min)
        ).clamp(min=dt_init_floor)
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)
        self.dt_proj.bias._no_reinit = True

        #parameter A
        A = repeat(
            torch.arange(1, self.d_state + 1, dtype=torch.float32, device=device),
            "n -> d n",
            d=self.d_inner,
        ).contiguous()
        A_log = torch.log(A)
        self.A_log = nn.Parameter(A_log)
        self.A_log._no_weight_decay = True

        #parameter D
        self.D = nn.Parameter(torch.ones(self.d_inner, device=device))
        self.D._no_weight_decay = True

        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=bias, **factory_kwargs)

    def forward(self, x_hsi, x_lidar): #x_hsi:   [Batch, 121, 64],x_lidar: [Batch, 121, 64] (Patch=11, Dim=64)
        B, L, _ = x_hsi.shape
        
        xz = self.in_proj(x_hsi) # [B, L, 128]
        x, z = xz.chunk(2, dim=-1) # x:[B, L, 64], z:[B, L, 64]

        x = x.transpose(1, 2) # [B, 64, L]
        x = self.conv1d(x)[:, :, :L] 
        x = F.silu(x) 
        x = x.transpose(1, 2) 

        x_dbl = self.x_proj(x)  # [B, L, dt_rank + 2*d_state]
        dt_raw, B_ssm, C_ssm = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
    
        dt_bias = self.lidar_modulator(x_lidar) # [B, L, dt_rank]
        
        dt_combined = dt_raw + dt_bias 
   
        dt = self.dt_proj(dt_combined) # [B, L, d_inner]
        
        x = rearrange(x, "b l d -> b d l")
        dt = rearrange(dt, "b l d -> b d l")
        B_ssm = rearrange(B_ssm, "b l dstate -> b dstate l").contiguous()
        C_ssm = rearrange(C_ssm, "b l dstate -> b dstate l").contiguous()
        z = rearrange(z, "b l d -> b d l")

        A = -torch.exp(self.A_log.float()) # [d_inner, d_state]

        y = selective_scan_fn(
            x,
            dt,
            A,
            B_ssm,
            C_ssm,
            self.D.float(),
            z=z, 
            delta_bias=None,    
            delta_softplus=True,
            return_last_state=None,
        )

        y = rearrange(y, "b d l -> b l d") # [B, L, 64]
        out = self.out_proj(y)             # [B, L, 64]
        
        return out
    
class Block(nn.Module):
    def __init__(self, 
             dim, 
             mlp_ratio=4.,         
             drop_path=0.,        
             layer_scale=1e-5      
             ):
        super().__init__()
        
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        
        self.mixer = CrossMambaMixer(
            d_model=dim, 
            d_state=16,  
            d_conv=4,    
            expand=2   
        )

        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = DynamicMlp(in_features=dim, hidden_features=mlp_hidden_dim)
        
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        
        self.gamma_1 = nn.Parameter(layer_scale * torch.ones(dim))
        self.gamma_2 = nn.Parameter(layer_scale * torch.ones(dim))

    def forward(self, x, x_lidar):
        x = x + self.drop_path(self.gamma_1 * self.mixer(self.norm1(x), x_lidar))
        
        x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))
    
        return x

class CrossMambaEncoder(nn.Module):
    def __init__(self, dim, depth, mlp_ratio=4., drop_path_rate=0.01):
        super().__init__()
        
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        
        self.layers = nn.ModuleList([
            Block(
                dim=dim,
                mlp_ratio=mlp_ratio,
                drop_path=dpr[i]
            )
            for i in range(depth)
        ])
        
        self.encoder_norm = nn.LayerNorm(dim, eps=1e-6)

    def forward(self, x, x_lidar):
        for layer_block in self.layers:
            x = layer_block(x, x_lidar)

        encoded = self.encoder_norm(x)
        return encoded


class CrossMamba(nn.Module):
    def __init__(self, FM, NC, NCLidar, Classes, HSIOnly, patchsize):
        super(CrossMamba, self).__init__()
        self.HSIOnly = HSIOnly
        self.patchsize = patchsize
        dim = FM * 4 
        
        self.conv5 = nn.Sequential(
            nn.Conv3d(1, 8, (9, 3, 3), padding=(0,1,1), stride=1),
            nn.BatchNorm3d(8),
            nn.ReLU()
        )
    
        input_channels = 8 * (NC - 8)
        self.conv6 = nn.Sequential(
            HetConv(input_channels, dim,
                p=1,
                g=dim//4 if input_channels % FM == 0 else dim//8,
            ),
            nn.BatchNorm2d(dim),
            nn.ReLU()
        )
        self.fft_enhancer = FrequencyEnhancer(dim=dim, h=patchsize, w=patchsize)
 
        self.lidarConv = nn.Sequential(
            nn.Conv2d(NCLidar, dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(dim),
            nn.GELU(),
            nn.Conv2d(dim, dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(dim),
            nn.GELU()
        )

        self.num_tokens = patchsize * patchsize 
        self.position_embeddings = nn.Parameter(torch.zeros(1, self.num_tokens, dim))
        trunc_normal_(self.position_embeddings, std=.02)

        self.pos_drop = nn.Dropout(p=0.1)

        self.encoder = CrossMambaEncoder(
            dim=dim, 
            depth=1,           
            mlp_ratio=4., 
            drop_path_rate=0.1
        )
        
        self.out3 = nn.Linear(dim, Classes)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out")
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x1, x2):  # x1: HSI, x2: LiDAR
        B = x1.shape[0]
        
        x1 = x1.unsqueeze(1) 
        x1 = self.conv5(x1) # -> [B, 8, C-8, H, W]
        x1 = x1.reshape(B, -1, self.patchsize, self.patchsize)
        x1 = self.conv6(x1) # ->[B, dim, H, W]

        if self.HSIOnly:
            x2_feat = torch.zeros_like(x1)
        else:
            x2_feat = self.lidarConv(x2) # [B, dim, H, W]
        
        x1 = self.fft_enhancer(x1, x2_feat) 
        
        # Flatten: [B, dim, H, W] -> [B, dim, L] -> [B, L, dim]
        x_hsi = x1.flatten(2).transpose(1, 2)
        x_hsi = x_hsi + self.position_embeddings
        x_hsi = self.pos_drop(x_hsi)

        x_lidar_token = x2_feat.flatten(2).transpose(1, 2)

        x_feat = self.encoder(x_hsi, x_lidar_token) 
        
        x_cls = x_feat.mean(dim=1)
        out = self.out3(x_cls)
        
        return out
  


batchsize = 128
patchsize = 5
model = CrossMamba(16, 144,1, 15, False, patchsize=patchsize).to("cuda")
summary(model, input_size=[
    (batchsize, 144, patchsize, patchsize),  
    (batchsize, 1, patchsize, patchsize)     
])



Layer (type:depth-idx)                        Output Shape              Param #
CrossMamba                                    [128, 15]                 1,600
├─Sequential: 1-1                             [128, 8, 136, 5, 5]       --
│    └─Conv3d: 2-1                            [128, 8, 136, 5, 5]       656
│    └─BatchNorm3d: 2-2                       [128, 8, 136, 5, 5]       16
│    └─ReLU: 2-3                              [128, 8, 136, 5, 5]       --
├─Sequential: 1-2                             [128, 64, 5, 5]           --
│    └─HetConv: 2-4                           [128, 64, 5, 5]           --
│    │    └─Conv2d: 3-1                       [128, 64, 5, 5]           39,232
│    │    └─Conv2d: 3-2                       [128, 64, 5, 5]           69,696
│    └─BatchNorm2d: 2-5                       [128, 64, 5, 5]           128
│    └─ReLU: 2-6                              [128, 64, 5, 5]           --
├─Sequential: 1-3                             [128, 64, 5, 5]           --
│    └─

# model training

In [4]:
DATASETS_WITH_HSI_PARTS = ['Berlin', 'Augsburg']
DATA2_List = ['SAR','DSM','MS']
# All datasets = "Houston","Trento","MUUFL","HoustonMS","AugsburgSAR","AugsburgDSM"
datasetNames = ["MUUFL"]

patchsize = 5
batchsize = 256
testSizeNumber = 500
EPOCH = 500
BandSize = 1
LR = 1e-3
FM = 16
HSIOnly = False
FileName = 'Geo-ssm0'
NUM_RUNS = 3

def load_dataset_file(file_path):
    try:
        return io.loadmat(file_path)
    except Exception:
        try:
            with h5py.File(file_path, 'r') as f:
                if 'Data' in f:
                    return {'Data': f['Data'][:]} 
                else:
                    keys = list(f.keys())
                    raise KeyError(f"在文件中找不到 'Data' 键。文件包含的键有: {keys}")
        except Exception as e:
            print(f"无法读取文件: {file_path}")
            raise e

def AA_andEachClassAccuracy(confusion_matrix):
    counter = confusion_matrix.shape[0]
    list_diag = np.diag(confusion_matrix)
    list_raw_sum = np.sum(confusion_matrix, axis=1)
    each_acc = np.nan_to_num(truediv(list_diag, list_raw_sum))
    average_acc = np.mean(each_acc)
    return each_acc, average_acc

def reports(xtest, xtest2, ytest, name, model):
    pred_y = np.empty((len(ytest)), dtype=np.float32)
    number = len(ytest) // testSizeNumber

    for i in range(number):
        temp = xtest[i * testSizeNumber:(i + 1) * testSizeNumber, :, :]
        temp = temp.cuda()
        temp1 = xtest2[i * testSizeNumber:(i + 1) * testSizeNumber, :, :]
        temp1 = temp1.cuda()

        temp2 = model(temp, temp1)
        
        temp3 = torch.max(temp2, 1)[1].squeeze()
        pred_y[i * testSizeNumber:(i + 1) * testSizeNumber] = temp3.cpu()
        del temp, temp2, temp3, temp1

    if (i + 1) * testSizeNumber < len(ytest):
        temp = xtest[(i + 1) * testSizeNumber:len(ytest), :, :]
        temp = temp.cuda()
        temp1 = xtest2[(i + 1) * testSizeNumber:len(ytest), :, :]
        temp1 = temp1.cuda()

        temp2 = model(temp, temp1)
        temp3 = torch.max(temp2, 1)[1].squeeze()
        pred_y[(i + 1) * testSizeNumber:len(ytest)] = temp3.cpu()
        del temp, temp2, temp3, temp1

    pred_y = torch.from_numpy(pred_y).long()
    
    oa = accuracy_score(ytest, pred_y)
    confusion = confusion_matrix(ytest, pred_y)
    each_acc, aa = AA_andEachClassAccuracy(confusion)
    kappa = cohen_kappa_score(ytest, pred_y)

    precision = precision_score(ytest, pred_y, average='macro', zero_division=0)
    recall = recall_score(ytest, pred_y, average='macro', zero_division=0)
    f1 = f1_score(ytest, pred_y, average='macro', zero_division=0)

    cls_report = classification_report(ytest, pred_y, digits=4)

    return confusion, oa*100, each_acc*100, aa*100, kappa*100, precision*100, recall*100, f1*100, cls_report

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)



def train():
    data_root = './dataset/' 
    for BandSize in [1]:
        for datasetName in datasetNames:
            print("----------------------------------Training for ", datasetName, " ---------------------------------------------")
            try:
                os.makedirs(datasetName)
            except FileExistsError:
                pass
            
            data1Name = ''
            data2Name = ''
            if datasetName in ["Houston", "Trento", "MUUFL"]:
                data1Name = datasetName
                data2Name = "LIDAR"
            else:
                for dataName in DATA2_List:
                    dataNameToCheck = re.compile(dataName)
                    matchObj = dataNameToCheck.search(datasetName)
                    if matchObj:
                        data1Name = datasetName.replace(dataName, "")
                        data2Name = dataName
            
            folder_suffix = str(patchsize) + 'x' + str(patchsize) 
            
            print(f"Loading data from folder: {data1Name + folder_suffix}")

            hsi_path = data_root + data1Name + folder_suffix + '/HSI_Tr.mat'
            print(f"Loading: {hsi_path}")
            HSI = load_dataset_file(hsi_path) 
            TrainPatch = HSI['Data'].astype(np.float32)
            NC = TrainPatch.shape[3]

            lidar_path = data_root + data1Name + folder_suffix + '/' + data2Name + '_Tr.mat'
            LIDAR = load_dataset_file(lidar_path) 
            TrainPatch2 = LIDAR['Data'].astype(np.float32)
            NCLIDAR = TrainPatch2.shape[3]

            label_path = data_root + data1Name + folder_suffix + '/TrLabel.mat'
            label = load_dataset_file(label_path) 
            TrLabel = label['Data']

            if data1Name in DATASETS_WITH_HSI_PARTS:
                i = 2
                basePath = data_root + data1Name + folder_suffix + '/HSI_Te_Part'
                part_data = load_dataset_file(basePath + str(i - 1) + '.mat')['Data']
                TestPatch = part_data
                while True:
                    my_file = Path(basePath + str(i) + '.mat')
                    if my_file.exists():
                        next_part = load_dataset_file(str(my_file))['Data'] 
                        TestPatch = np.concatenate([TestPatch, next_part], axis=0)
                        i += 1
                    else:
                        break
            else:
                hsi_te_path = data_root + data1Name + folder_suffix + '/HSI_Te.mat'
                HSI = load_dataset_file(hsi_te_path) 
                TestPatch = HSI['Data']
            
            TestPatch = TestPatch.astype(np.float32)

            lidar_te_path = data_root + data1Name + folder_suffix + '/' + data2Name + '_Te.mat'
            LIDAR = load_dataset_file(lidar_te_path) 
            TestPatch2 = LIDAR['Data'].astype(np.float32)

            label_te_path = data_root + data1Name + folder_suffix + '/TeLabel.mat'
            label = load_dataset_file(label_te_path) 
            TsLabel = label['Data']

            TrainPatch1 = torch.from_numpy(TrainPatch).to(torch.float32).permute(0, 3, 1, 2)
            TrainPatch2 = torch.from_numpy(TrainPatch2).to(torch.float32).permute(0, 3, 1, 2)
            TrainLabel1 = torch.from_numpy(TrLabel) - 1
            TrainLabel1 = TrainLabel1.long().reshape(-1)

            TestPatch1 = torch.from_numpy(TestPatch).to(torch.float32).permute(0, 3, 1, 2)
            TestPatch2 = torch.from_numpy(TestPatch2).to(torch.float32).permute(0, 3, 1, 2)
            TestLabel1 = torch.from_numpy(TsLabel) - 1
            TestLabel1 = TestLabel1.long().reshape(-1)

            Classes = len(np.unique(TrainLabel1))

            print("Move Training Data to GPU...")
            TrainPatch1 = TrainPatch1.cuda()
            TrainPatch2 = TrainPatch2.cuda()
            TrainLabel1 = TrainLabel1.cuda()

            dataset = dataf.TensorDataset(TrainPatch1, TrainPatch2, TrainLabel1)
            
            train_loader = dataf.DataLoader(dataset, batch_size=batchsize, shuffle=True, num_workers=0)

            print("HSI Train data shape = ", TrainPatch1.shape)
            print("Train label shape = ", TrainLabel1.shape)
            print("Number of Classes = ", Classes)

            classes_count = torch.bincount(TrainLabel1.cpu())
            weights = 1.0 / torch.sqrt(classes_count.float())

            weights[torch.isinf(weights)] = 0 
            
            weights = weights / weights.mean()
            
            class_weights = weights.cuda()
            
            print(f"Dataset: {datasetName}")
            print("Smoothed Class Weights:", class_weights)

            OA = []
            AA = []
            PRECISION = [] 
            KAPPA = [] 
            F1 = []        
            ELEMENT_ACC = np.zeros((NUM_RUNS, Classes))

            set_seed(42)

            for iterNum in range(NUM_RUNS):
                model = CrossMamba(FM, NC, NCLIDAR, Classes, HSIOnly, patchsize=patchsize).cuda()               
                optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.005)
                loss_func = DualModalBoundaryLoss(alpha=0.5, beta=0.5, weight=class_weights).cuda()
                scheduler = CosineAnnealingLR(optimizer, T_max=EPOCH, eta_min=1e-6) 
                
                BestAcc = 0
                torch.cuda.synchronize()
                start = time.time()

                print(f"Start Training Run {iterNum+1}/{NUM_RUNS}...")

                for epoch in range(EPOCH):
                    model.train() 
                    epoch_loss_sum = 0.0
                    
                    for step, (b_x1, b_x2, b_y) in enumerate(train_loader):                        
                        optimizer.zero_grad()
                        
                        if HSIOnly:
                            out1 = model(b_x1, b_x2)
                            loss = loss_func(out1, b_y)
                        else:
                            out = model(b_x1, b_x2)
                            loss = loss_func(out, b_y, b_x1, b_x2)

                        loss.backward()
                        optimizer.step()
                        
                        epoch_loss_sum += loss.item()
                    
                    avg_loss = epoch_loss_sum / len(train_loader)
                    
                    model.eval()
                    pred_y_list = []
                    
                    with torch.no_grad(): 
                        number = len(TestLabel1) // testSizeNumber
                        for i in range(number):
                            temp = TestPatch1[i * testSizeNumber:(i + 1) * testSizeNumber].cuda()
                            temp1 = TestPatch2[i * testSizeNumber:(i + 1) * testSizeNumber].cuda()
                            
                            out_test = model(temp, temp1)
                            pred = torch.max(out_test, 1)[1]
                            pred_y_list.append(pred.cpu())
                        
                        if (i + 1) * testSizeNumber < len(TestLabel1):
                            temp = TestPatch1[(i + 1) * testSizeNumber:].cuda()
                            temp1 = TestPatch2[(i + 1) * testSizeNumber:].cuda()
                            
                            out_test = model(temp, temp1)
                            pred = torch.max(out_test, 1)[1]
                            pred_y_list.append(pred.cpu())
                    
                    pred_y_full = torch.cat(pred_y_list).long()
                    
                    accuracy = torch.sum(pred_y_full == TestLabel1).item() / TestLabel1.size(0)

                    print('Epoch: %3d | train loss: %.4f | test accuracy: %.4f' % (epoch, avg_loss, accuracy * 100))

                    if accuracy > BestAcc:
                        BestAcc = accuracy
                        torch.save(model.state_dict(), datasetName + '/net_params_' + FileName + '.pkl')

                    scheduler.step()

                torch.cuda.synchronize()
                end = time.time()
                print(f"Run {iterNum+1} Time: {end - start:.2f}s")
                
                model.load_state_dict(torch.load(datasetName + '/net_params_' + FileName + '.pkl'))
                model.eval()
                
                if iterNum == 0:
                    dummy_hsi = torch.randn(1, NC, patchsize, patchsize).cuda() 
                    dummy_lidar = torch.randn(1, NCLIDAR, patchsize, patchsize).cuda()
                    try:
                        macs, params = profile(model, inputs=(dummy_hsi, dummy_lidar), verbose=False)
                        print(f"Params: {params / 1e6:.4f} M, FLOPs: {macs * 2 / 1e6:.4f} M")
                    except:
                        print("Thop profile failed, skipping.")

                confusion, oa, each_acc, aa, kappa, precision, recall, f1, cls_report = reports(TestPatch1, TestPatch2, TestLabel1, datasetName, model)
                
                print("\n╔════════════════════════════════════╗")
                print("║         Per-Class Accuracy         ║")
                print("╠════════════════════════════════════╣")

                if datasetName == 'Trento':
                    class_names = ['Apples', 'Buildings', 'Ground', 'Woods', 'Vineyard', 'Roads']
                elif datasetName == 'Houston':
                    class_names = ['Healthy grass', 'Stressed grass', 'Synthetic grass', 'Trees', 
                                   'Soil', 'Water', 'Residential', 'Commercial', 'Road', 'Highway', 
                                   'Railway', 'Parking Lot 1', 'Parking Lot 2', 'Tennis Court', 'Running Track']
                elif datasetName == 'MUUFL':
                    class_names = ['Trees','Grass_Pure','Grass_Groundsurface','Dirt_And_Sand', 'Road_Materials','Water',"Buildings'_Shadow",
                    'Buildings','Sidewalk','Yellow_Curb','ClothPanels']
                else:
                    class_names = [f"Class {i+1}" for i in range(len(each_acc))]
                
                for i, acc in enumerate(each_acc):
                    name = class_names[i] if i < len(class_names) else f"Class {i+1}"
                    print(f"║ {name:<25} : {acc:6.2f} % ║")
                print("╚════════════════════════════════════╝\n")
                
                OA.append(oa)
                AA.append(aa)
                KAPPA.append(kappa)  
                PRECISION.append(precision) 
                F1.append(f1)               
                ELEMENT_ACC[iterNum, :] = each_acc
                
                torch.save(model, datasetName + '/best_model_' + FileName + '_BandSize' + str(BandSize) + '_Iter' + str(iterNum) + '.pt')

                print(f"Run {iterNum+1} Results -> OA: {oa:.2f}, AA: {aa:.2f}, Kappa: {kappa:.2f}, PR:{precision:.2f}, F1: {f1:.2f}")

                print("----------" + datasetName + " Training Finished -----------")
                
            record.record_output(OA, AA, KAPPA, ELEMENT_ACC, PRECISION, F1, './' + datasetName + '/' + FileName + '_BandSize' + str(BandSize) + '_Report_' + datasetName + '.txt')

if __name__ == '__main__':
    train()

----------------------------------Training for  MUUFL  ---------------------------------------------
Loading data from folder: MUUFL5x5
Loading: ./dataset/MUUFL5x5/HSI_Tr.mat


NameError: name 'MinMaxScaler' is not defined